In [2]:
import pandas as pd
import numpy as np


In [3]:
df=pd.read_csv("TMDB_Movies_Dataset.csv",usecols=['id','title','overview','vote_count','vote_average','genre_names'])


In [4]:
df.head(2)


,id,title,overview,vote_count,vote_average,genre_names
0,1523145,Your Heart Will Be Broken,High school student Polina is saved from bully...,24,6.896,"Romance,Drama"
1,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,2066,7.300,"Science Fiction,Adventure,Fantasy"


In [5]:
df.shape


(10000, 6)

In [6]:
(df.isnull().sum()/df.shape[0])*100 # percentages of the null values in each column


id              0.00
title           0.00
overview        0.38
vote_count      0.00
vote_average    0.00
genre_names     0.51
dtype: float64

In [7]:
df.dropna(inplace=True)


In [8]:
(df.duplicated().sum()/df.shape[0])*100 # percentage of the duplicated values in the entire dataset


10.962081484469543

In [9]:
df.drop_duplicates(inplace=True)


In [10]:
df.reset_index(inplace=True) # reseting the index because if we don't do it then the dataset can contains missing indexes because of dropping the rows earlier


In [11]:
df['genre_names']=df['genre_names'].str.replace(',',' ') # replacing the comma with an empty space to prevent multiple words become one and as a result, a new word will get generated


In [12]:
df['genre_names']


0                                Romance Drama
1            Science Fiction Adventure Fantasy
2                 Comedy Science Fiction Crime
3                        Action Thriller Music
4                      Animation Comedy Family
                         ...                  
8824                    Drama History Thriller
8825       Adventure Drama Action Thriller War
8826                                    Comedy
8827                    Mystery Drama Thriller
8828    Horror Action Thriller Science Fiction
Name: genre_names, Length: 8829, dtype: object

In [13]:
df['combined_text']=df['title'] + " " + df['overview'] + " " + df['genre_names'] # concatenating the three columns that we will gonna use to make content based recommendations


In [14]:
df.head(3)


,index,id,title,overview,vote_count,vote_average,genre_names,combined_text
0,0,1523145,Your Heart Will Be Broken,High school student Polina is saved from bully...,24,6.896,Romance Drama,Your Heart Will Be Broken High school student ...
1,1,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,2066,7.300,Science Fiction Adventure Fantasy,Avatar: Fire and Ash In the wake of the devast...
2,2,1115544,Mike & Nick & Nick & Alice,Two gangsters and the woman they love try to s...,107,6.706,Comedy Science Fiction Crime,Mike & Nick & Nick & Alice Two gangsters and t...


1) lower
2) punctuation removal
3) lemmatization or stemming (root form of words)
4) tokenization
5) vectorization + stopwords(English) removal
6) Cosine similarity

In [15]:
df['combined_text']=df['combined_text'].str.lower() # Changing every single alphabet from UpperCase to LowerCase


In [16]:
import string
string.punctuation


'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [17]:
def punc_remover(text):
    text=''.join(ch if ch not in string.punctuation else'' for ch in text)
    return text


In [18]:
df['combined_text']=df['combined_text'].apply(punc_remover)
df['combined_text']


0       your heart will be broken high school student ...
1       avatar fire and ash in the wake of the devasta...
2       mike  nick  nick  alice two gangsters and the ...
3       pretty lethal a troupe of ballerinas find them...
4       goat a small goat with big dreams gets a oncei...
                              ...                        
8824    operation finale in 1960 a team of israeli sec...
8825    the eagle has landed when the nazi high comman...
8826    the full monty sheffield england gaz a jobless...
8827    rage a man brutally murders a married couple a...
8828    the darkest hour in moscow five young people l...
Name: combined_text, Length: 8829, dtype: object

In [19]:
patt=r'[^\w\s]'
df['combined_text']=df['combined_text'].str.replace(patt,'',regex=True)


In [20]:
df['combined_text'][0]


'your heart will be broken high school student polina is saved from bullying at her new school and makes a deal with the main bully bars he must pretend to be her boyfriend and protect her and she must do everything he says during this game the couple develops real feelings but her family and classmates have reasons to separate the lovers romance drama'

In [21]:
dirty_rows= df[df['combined_text'].str.contains(patt,regex=True)]
print(f"Rows still containing the punctuation:  {len(dirty_rows)}")


Rows still containing the punctuation:  0


In [22]:
# Lemmatization

import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer


In [23]:
lemmatizer=WordNetLemmatizer()


In [24]:
def my_lemmatizer(text):
    tokens=nltk.word_tokenize(text)
    return [lemmatizer.lemmatize(w) for w in tokens]


In [25]:
vectorizer=CountVectorizer(max_features=5000,tokenizer=my_lemmatizer,stop_words='english')


In [26]:
vectors=vectorizer.fit_transform(df['combined_text']).toarray()


c:\Users\SMART COM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\SMART COM\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


In [27]:
vectors


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [28]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vectors)


In [29]:
def recommend(movie):
    movie_ind= df[df['title']==movie].index[0] # get index of the input movie
    distance= similarity[movie_ind] # calculates the distance of the input movie at the index present in the dataset
    movies_list=sorted(list(enumerate(distance)),reverse=True,key= lambda x: x[1])[1:6] # to get the top 5 movie's sorted based on the distance along with the movie's index
    
    for i in movies_list:
        print(df.iloc[i[0]].id,df.iloc[i[0]].title) # it basically get the title of the movie by using the index that we get by using the enumerate function


In [30]:
df[['title','genre_names']].sample(4)


,title,genre_names
5420,Colour Blossoms,Romance
8452,Tales from the Dark 2,Horror
1391,Alice in Wonderland,Animation Family Fantasy Adventure
5451,Samson,Action Drama Adventure


In [31]:
recommend('Ice Age')


57800 Ice Age: Continental Drift
774825 The Ice Age Adventures of Buck Wild
421725 Scrat: Spaced Out
278154 Ice Age: Collision Course
1424965 Ice Skater


In [ ]:
import pickle as pk


In [ ]:
pk.dump(similarity,open("similarity.pkl",'wb'))
pk.dump(recommend,open("recommend.pkl",'wb'))
